In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))

In [ ]:
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))

# 검색 에이전트

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

web_search = TavilySearch(
    max_result = 5,
    topic = "general"
) 

llm = init_chat_model("gpt-5-nano")
search_agent = create_agent(
    llm,
    [web_search]
)

##파일 시스템 에이전트

In [ ]:
from langchain_core.tools import tool

@tool
def save_file(filename: str, content: str) -> str:
    """주어진 내용을 지정된 파일명으로 현재 폴더에 저장합니다."""
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    return f"'{filename}' 파일이 성공적으로 저장되었습니다."

In [ ]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("gpt-5-nano")

save_agent = create_agent(
    llm, [save_file]
)

## supervisor

In [ ]:
supervisor_system_message = """당신은 작업을 관리하는 Supervisor입니다. 
사용자의 요청을 분석하고 적절한 에이전트를 선택해야 합니다.

사용 가능한 에이전트:
1. Search: 웹 검색이 필요한 질문이나 최신 정보가 필요한 경우
2. Editor: 파일 저장이나 편집이 필요한 경우

사용자의 요청에 따라 다음 중 하나를 선택하세요:
- 검색이 필요한 경우: "Search"
- 파일 저장/편집이 필요한 경우: "Editor"
- 일반 대화나 작업 완료인 경우: "END"

응답 형식:
1. 먼저 사용자 요청을 분석하고 설명
2. 선택한 에이전트와 그 이유를 명시
3. 마지막에 반드시 "NEXT_AGENT: [선택한_에이전트]" 형태로 명시
4. Search 가 나온 경우에는 최소 검색을 1번은 진행하세요.
5. 계속해서 검색을 해도 나오지 않는 경우 END로 진행
"""

In [ ]:
from langgraph.prebuilt.chat_agent_executor import AgentState

def supervisor(state: AgentState):
    """Supervisor 노드: 사용자 요청을 분석하고 적절한 에이전트를 선택"""
    messages = [SystemMessage(content=supervisor_system_message)] + state["messages"]
    response = llm.invoke(messages)

    # 응답에서 다음 에이전트 추출
    next_agent = "END"  # 기본값
    if "NEXT_AGENT:" in response.content:
        try:
            next_agent = response.content.split("NEXT_AGENT:")[-1].strip()
            if next_agent not in ["Search", "Editor", "END"]:
                next_agent = "END"
        except:
            next_agent = "END"

    return {"messages": [response], "next_agent": next_agent}

In [ ]:
def route_agent(state:AgentState):
    return state.get("next_agent","END")

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
from langgraph.graph import START, END, StateGraph

In [ ]:
def save_node(state: AgentState):
    # 현재 상태(state)를 에이전트에게 전달하여 실행
    response = save_agent.invoke(state)
    
    # 결과 메시지 목록 중 가장 마지막 메시지 추출
    last_message = response["messages"][-1]
    
    # 마지막 메시지만 리스트에 담아 반환 (AgentState 업데이트)
    return {"messages": [last_message]}

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("Supervisor",supervisor)

#workflow.add_node("Search",search_agent)
#workflow.add_node("Editor",save_agent)

workflow.add_node("Search",search_agent)
workflow.add_node("Editor",save_node)

workflow.add_edge(START, "Supervisor")
workflow.add_conditional_edges(
    "Supervisor",
    route_agent,
    {
        "Search":"Search",
        "Editor":"Editor",
        "END":END
    }
)

workflow.add_edge("Search","Supervisor")
workflow.add_edge("Editor","Supervisor")
workflow.add_edge("Supervisor",END)

In [ ]:
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [ ]:
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# 마크다운 코드를 직접 표시한다. 더 선명하게 보인다.
from IPython.display import display, Markdown

# Get the Mermaid code
mermaid_code = app.get_graph().draw_mermaid()

# Display in Jupyter
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

In [ ]:
import uuid

thread_id = uuid.uuid4()
thread_id

In [ ]:
config = {'configurable': {'thread_id': thread_id}}
user_input = {
    'messages': [('user', '2026년 EPL 우승팀을 검색해줘')]
}

events = app.stream(user_input, config=config, stream_mode='values')
for event in events:
    event['messages'][-1].pretty_print()

In [ ]:
from langchain_core.messages import SystemMessage